[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/28_neural_networks/90_exercises.ipynb)

# Bài tập — Chương 28: Neural Networks (L0 → L7)

> Bài tập lập trình hàm kích hoạt theo khoá AIVN nằm riêng ở `91_assignment_activation_functions`.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/28_neural_networks

In [ ]:
import copy
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

rng = np.random.default_rng(0)
X_dg, y_dg = load_digits(return_X_y=True)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Đạo hàm của sigmoid, tanh, ReLU; vì sao sigmoid gây vanishing gradient? (b) Phương sai khởi tạo của Xavier và He; dùng với hàm kích hoạt nào? (c) Batch Norm tính gì khi train và khi eval? (d) Công thức cập nhật Adam. (e) Inverted dropout khi train và khi eval.

✍️ Trả lời:

**Đáp án:**
(a) $\sigma' = \sigma(1-\sigma) \le 0.25$; $\tanh' = 1 - \tanh^2 \le 1$; $\text{ReLU}' = \mathbb 1[x > 0]$. Backprop nhân các đạo hàm qua từng lớp; với sigmoid mỗi lớp nhân ≤ 0.25 (và ≈ 0 khi bão hoà) → gradient các lớp đầu tắt dần theo hàm mũ.
(b) Xavier: $\text{Var}(w) = 1/n_{in}$ (hoặc $2/(n_{in} + n_{out})$) cho tanh/sigmoid; He: $2/n_{in}$ cho ReLU (bù việc ReLU bỏ nửa âm).
(c) Train: chuẩn hoá bằng trung bình/phương sai của **batch**, cập nhật running mean/var; eval: dùng running mean/var; sau đó $\gamma\hat z + \beta$.
(d) $m \leftarrow \beta_1 m + (1-\beta_1)g$, $s \leftarrow \beta_2 s + (1-\beta_2)g^2$, $\hat m = m/(1-\beta_1^t)$, $\hat s = s/(1-\beta_2^t)$, $\theta \leftarrow \theta - \eta\hat m/(\sqrt{\hat s} + \epsilon)$.
(e) Train: nhân mặt nạ Bernoulli(1 − p) rồi chia $1 - p$; eval: không làm gì.

## L1 — Giải thích bằng lời

**Câu 2.** Một mạng 20 lớp ẩn, kích hoạt sigmoid, khởi tạo trọng số $\mathcal N(0, 1)$, SGD lr = 0.01, đầu vào là pixel 0–255 chưa chuẩn hoá. Loss đứng yên ở $\ln 10$. Đồng nghiệp đề xuất "tăng learning rate lên 100 lần". Phân tích nguyên nhân và đề xuất sửa theo thứ tự ưu tiên.

✍️ Trả lời:

**Gợi ý:** Loss = ln 10 nghĩa là mạng đoán đều 10 lớp — gradient gần như không tới được các lớp đầu. Nguyên nhân chồng chất: (1) đầu vào 0–255 + trọng số $\mathcal N(0, 1)$ → tiền kích hoạt cỡ hàng trăm → sigmoid **bão hoà** hoàn toàn (đạo hàm ≈ 0); (2) 20 lớp sigmoid nhân các đạo hàm ≤ 0.25 → vanishing gradient. Tăng LR không cứu được gradient bằng 0, còn có thể làm phân kỳ ở lớp cuối. Sửa: chuẩn hoá đầu vào (chia 255 hoặc chuẩn hoá theo kênh); đổi sang ReLU/GELU + khởi tạo He; thêm BatchNorm (hoặc LayerNorm); cân nhắc ít lớp hơn hoặc kết nối tắt (residual); dùng Adam(W) với LR range test; kiểm tra bằng cách overfit một batch nhỏ và theo dõi chuẩn gradient từng lớp.

## L2 — Tính tay

**Câu 3.** Adam với $\eta = 0.1$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 0$, $\theta_0 = 1$. Gradient hai bước: $g_1 = 2$, $g_2 = -1$. Tính $\hat m_1, \hat s_1, \theta_1$, rồi $\theta_2$. Nhận xét độ lớn bước đầu tiên.

In [ ]:
### START CODE HERE ### (≈ 10 dòng)
m1, s1 = 0.1 * 2, 0.001 * 4
m_hat1, s_hat1 = m1 / (1 - 0.9), s1 / (1 - 0.999)
theta1 = 1 - 0.1 * m_hat1 / math.sqrt(s_hat1)
m2, s2 = 0.9 * m1 + 0.1 * (-1), 0.999 * s1 + 0.001 * 1
m_hat2, s_hat2 = m2 / (1 - 0.9**2), s2 / (1 - 0.999**2)
theta2 = theta1 - 0.1 * m_hat2 / math.sqrt(s_hat2)
### END CODE HERE ###
print(f"m̂₁ = {m_hat1:.4f}, ŝ₁ = {s_hat1:.4f}, θ₁ = {theta1:.4f}, θ₂ = {theta2:.4f}")

In [ ]:
p = torch.tensor([1.0], dtype=torch.float64, requires_grad=True)
opt3 = torch.optim.Adam([p], lr=0.1, betas=(0.9, 0.999), eps=0.0)
for g in [2.0, -1.0]:
    opt3.zero_grad()
    (p * g).sum().backward()                                            # gradient theo p đúng bằng g
    opt3.step()
check_close(m_hat1, 2.0, message="m̂₁ = g₁ (hiệu chỉnh bias)")
check_close(theta1, 0.9, message="bước đầu có độ lớn đúng bằng η")
check_close(theta2, p.item(), tol=1e-10, message="θ₂ khớp torch.optim.Adam")

**Nhận xét:** Nhờ hiệu chỉnh bias, ở bước đầu $\hat m_1/\sqrt{\hat s_1} = g_1/|g_1| = \pm1$ → bước có độ lớn đúng bằng $\eta$ **bất kể** độ lớn gradient — Adam gần như "bất biến theo thang đo gradient". Không hiệu chỉnh, $m_1 = 0.2$, $\sqrt{s_1} = 0.063$ → bước ≈ $3.2\eta$ thay vì $\eta$.

**Câu 4.** (a) Lớp có $n_{in} = 256$, đầu vào $\text{Var}(x) = 1$, trung bình 0. Độ lệch chuẩn khởi tạo để $\text{Var}(z) = 1$ (Xavier)? He (cho ReLU)? (b) Batch Norm (không $\gamma, \beta$, $\epsilon = 0$) trên batch $[1, 2, 3, 6]$ của một feature: đầu ra?

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
std_xavier = math.sqrt(1 / 256)
std_he = math.sqrt(2 / 256)
batch4 = np.array([1.0, 2.0, 3.0, 6.0])
bn4 = (batch4 - batch4.mean()) / batch4.std()
### END CODE HERE ###
print(f"Xavier std = {std_xavier:.4f} | He std = {std_he:.4f} | BN → {bn4.round(4)}")

In [ ]:
check_close(std_xavier, 0.0625, message="Xavier")
check_close(std_he, math.sqrt(2) / 16, message="He")
check_close(bn4, (batch4 - 3) / math.sqrt(3.5), message="BN dùng phương sai chia n")
check_close(nn.BatchNorm1d(1, affine=False, eps=1e-12)(torch.tensor(batch4[:, None], dtype=torch.float32)).detach().numpy().ravel(), bn4, tol=1e-6, message="khớp nn.BatchNorm1d (train)")

## L3 — Code

**Bài 5.** Cài đặt optimizer NumPy dạng lớp, cập nhật **tại chỗ** một danh sách tham số:
- `MomentumSGD(lr, momentum)` — kiểu PyTorch: $v \leftarrow \mu v + g$, $\theta \leftarrow \theta - \eta v$;
- `Adam(lr, betas, eps)`.

Phương thức `step(params, grads)`. Kiểm tra trên hồi quy tuyến tính: sau 50 bước, tham số khớp `torch.optim` (cùng khởi tạo, cùng gradient).

In [ ]:
class MomentumSGD:
    def __init__(self, lr=0.01, momentum=0.9):
        self.lr, self.momentum, self.v = lr, momentum, None

    def step(self, params, grads):
        ### START CODE HERE ### (≈ 5 dòng)
        # STARTER: # khởi tạo self.v = [zeros_like(p)] lần đầu; với mỗi (p, g, v): v = momentum·v + g; p -= lr·v (tại chỗ)
        if self.v is None:
            self.v = [np.zeros_like(p) for p in params]
        for p, g, v in zip(params, grads, self.v):
            v *= self.momentum
            v += g
            p -= self.lr * v
        ### END CODE HERE ###


class Adam:
    def __init__(self, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.lr, (self.b1, self.b2), self.eps = lr, betas, eps
        self.m = self.s = None
        self.t = 0

    def step(self, params, grads):
        ### START CODE HERE ### (≈ 10 dòng)
        # STARTER: # lần đầu: m, s = zeros; t += 1; m = b1·m + (1−b1)·g; s = b2·s + (1−b2)·g²
        # STARTER: # hiệu chỉnh bias, p -= lr · m̂ / (√ŝ + eps) — cập nhật tại chỗ
        if self.m is None:
            self.m = [np.zeros_like(p) for p in params]
            self.s = [np.zeros_like(p) for p in params]
        self.t += 1
        for p, g, m, s in zip(params, grads, self.m, self.s):
            m *= self.b1
            m += (1 - self.b1) * g
            s *= self.b2
            s += (1 - self.b2) * g**2
            m_hat, s_hat = m / (1 - self.b1**self.t), s / (1 - self.b2**self.t)
            p -= self.lr * m_hat / (np.sqrt(s_hat) + self.eps)
        ### END CODE HERE ###

In [ ]:
X5 = rng.normal(size=(64, 5))
y5 = X5 @ rng.normal(size=5) + 0.1 * rng.normal(size=64)
for MyOpt, TorchOpt, kw_mine, kw_torch in [(MomentumSGD, torch.optim.SGD, {"lr": 0.01, "momentum": 0.9}, {"lr": 0.01, "momentum": 0.9}),
                                           (Adam, torch.optim.Adam, {"lr": 0.05}, {"lr": 0.05})]:
    w_np, b_np = np.zeros(5), np.zeros(1)
    opt_np = MyOpt(**kw_mine)
    w_t = torch.zeros(5, dtype=torch.float64, requires_grad=True)
    b_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt_t = TorchOpt([w_t, b_t], **kw_torch)
    for _ in range(50):
        err = X5 @ w_np + b_np - y5
        opt_np.step([w_np, b_np], [2 * X5.T @ err / len(y5), np.array([2 * err.mean()])])
        opt_t.zero_grad()
        ((torch.tensor(X5) @ w_t + b_t - torch.tensor(y5)) ** 2).mean().backward()
        opt_t.step()
    check_close(w_np, w_t.detach().numpy(), tol=1e-9, message=f"{MyOpt.__name__}: trọng số khớp torch")
    check_close(b_np, b_t.detach().numpy(), tol=1e-9, message=f"{MyOpt.__name__}: bias khớp torch")

**Bài 6.** Viết `init_weight(fan_in, fan_out, kind, rng)` với `kind ∈ {"xavier", "he"}` (phân phối chuẩn, trung bình 0). Truyền một batch chuẩn tắc qua 30 lớp × 256 (tanh với Xavier, ReLU với He) và kiểm tra độ lệch chuẩn kích hoạt ở lớp 30 không tắt dần/bùng nổ; so với ReLU + Xavier.

In [ ]:
def init_weight(fan_in, fan_out, kind, rng):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return None
    std = math.sqrt(1 / fan_in) if kind == "xavier" else math.sqrt(2 / fan_in)
    return rng.normal(0, std, (fan_in, fan_out))
    ### END CODE HERE ###


def final_std(act, kind, depth=30, width=256, seed=0):
    r = np.random.default_rng(seed)
    h = r.normal(size=(500, width))
    for _ in range(depth):
        h = act(h @ init_weight(width, width, kind, r))
    return h.std()

In [ ]:
relu = lambda z: np.maximum(0, z)
s_tanh_xavier, s_relu_he, s_relu_xavier = final_std(np.tanh, "xavier"), final_std(relu, "he"), final_std(relu, "xavier")
print(f"std ở lớp 30: tanh + Xavier = {s_tanh_xavier:.3f} | ReLU + He = {s_relu_he:.3f} | ReLU + Xavier = {s_relu_xavier:.2e}")
check_close(np.std(init_weight(1000, 1000, "he", np.random.default_rng(1))), math.sqrt(2 / 1000), tol=1e-3, message="He std")
check_true(0.3 < s_relu_he < 3, "ReLU + He giữ tín hiệu ổn định qua 30 lớp")
check_true(s_tanh_xavier > 0.05, "tanh + Xavier không tắt hẳn")
check_less(s_relu_xavier, 0.01, "ReLU + Xavier tắt dần theo hàm mũ")

## L4 — Debug

**Bài 7.** Mạng dưới đây (digits) học kém (accuracy validation chỉ ~0.83): (1) kích hoạt sigmoid ở 8 lớp ẩn; (2) khởi tạo $\mathcal N(0, 1)$; (3) đầu vào pixel 0–16 không chuẩn hoá. Sửa: ReLU + khởi tạo He (bias 0) + `StandardScaler` (fit trên train). Giữ nguyên số lớp, độ rộng, optimizer, số epoch. Lưu `val_acc7`.

In [ ]:
X7_tr, X7_va, y7_tr, y7_va = train_test_split(X_dg, y_dg, test_size=0.3, random_state=0, stratify=y_dg)
### START CODE HERE ### (≈ 16 dòng)
# STARTER: Xt, Xv = torch.tensor(X7_tr, dtype=torch.float32), torch.tensor(X7_va, dtype=torch.float32)   # 🐛 (3)
# STARTER: torch.manual_seed(0)
# STARTER: layers, d = [], 64
# STARTER: for _ in range(8):
# STARTER:     lin = nn.Linear(d, 64)
# STARTER:     nn.init.normal_(lin.weight, 0, 1.0)                                                              # 🐛 (2)
# STARTER:     layers += [lin, nn.Sigmoid()]                                                                    # 🐛 (1)
# STARTER:     d = 64
sc7 = StandardScaler().fit(X7_tr)
Xt, Xv = (torch.tensor(sc7.transform(a), dtype=torch.float32) for a in (X7_tr, X7_va))
torch.manual_seed(0)
layers, d = [], 64
for _ in range(8):
    lin = nn.Linear(d, 64)
    nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
    nn.init.zeros_(lin.bias)
    layers += [lin, nn.ReLU()]
    d = 64
### END CODE HERE ###
model7 = nn.Sequential(*layers, nn.Linear(d, 10))
yt, yv = torch.tensor(y7_tr), torch.tensor(y7_va)
opt7 = torch.optim.SGD(model7.parameters(), lr=0.02, momentum=0.9)
g7 = torch.Generator().manual_seed(0)
for epoch in range(20):
    perm = torch.randperm(len(Xt), generator=g7)
    for s in range(0, len(perm), 32):
        idx = perm[s:s + 32]
        opt7.zero_grad()
        nn.functional.cross_entropy(model7(Xt[idx]), yt[idx]).backward()
        opt7.step()
with torch.no_grad():
    val_acc7 = (model7(Xv).argmax(1) == yv).float().mean().item()
print(f"val accuracy = {val_acc7:.3f}")

In [ ]:
check_greater(val_acc7, 0.94, "mạng 8 lớp sau khi sửa: val accuracy > 0.94")
check_true(not any(isinstance(m, nn.Sigmoid) for m in model7.modules()), "không còn sigmoid ở lớp ẩn")

## L5 — Phân tích

**Bài 8.** **Batch size và learning rate.** MLP 64 → 128 → 10 trên digits, SGD momentum 0.9, **20 epoch** cố định. Với batch size $B \in \{8, 32, 128, 512\}$: (a) giữ lr = 0.01; (b) **tỉ lệ tuyến tính** lr = 0.01 · B/8. Lưu `bs8[B] = {"steps", "loss_fixed", "acc_fixed", "loss_scaled", "acc_scaled"}` (loss trên train, accuracy trên validation 30%). Nhận xét.

In [ ]:
X8_tr, X8_va, y8_tr, y8_va = train_test_split(X_dg, y_dg, test_size=0.3, random_state=0, stratify=y_dg)
sc8 = StandardScaler().fit(X8_tr)
Xt8, Xv8 = (torch.tensor(sc8.transform(a), dtype=torch.float32) for a in (X8_tr, X8_va))
yt8, yv8 = torch.tensor(y8_tr), torch.tensor(y8_va)


def train8(batch_size, lr, epochs=20, seed=0):
    ### START CODE HERE ### (≈ 14 dòng)
    # STARTER: # mô hình nn.Sequential(Linear(64,128), ReLU, Linear(128,10)), torch.manual_seed(seed); SGD momentum 0.9
    # STARTER: # duyệt epoch × mini-batch (randperm với Generator seed), đếm số bước; trả về (loss train, accuracy val, số bước)
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    g = torch.Generator().manual_seed(seed)
    steps = 0
    for _ in range(epochs):
        perm = torch.randperm(len(Xt8), generator=g)
        for s in range(0, len(perm), batch_size):
            idx = perm[s:s + batch_size]
            opt.zero_grad()
            nn.functional.cross_entropy(model(Xt8[idx]), yt8[idx]).backward()
            opt.step()
            steps += 1
    with torch.no_grad():
        return nn.functional.cross_entropy(model(Xt8), yt8).item(), (model(Xv8).argmax(1) == yv8).float().mean().item(), steps
    ### END CODE HERE ###


bs8 = {}
for B in [8, 32, 128, 512]:
    lf, af, steps = train8(B, 0.01)
    ls, as_, _ = train8(B, 0.01 * B / 8)
    bs8[B] = {"steps": steps, "loss_fixed": lf, "acc_fixed": af, "loss_scaled": ls, "acc_scaled": as_}
    print(f"B = {B:3d} ({steps:4d} bước): lr cố định → loss {lf:.4f}, acc {af:.3f} | lr tỉ lệ ({0.01 * B / 8:.2f}) → loss {ls:.4f}, acc {as_:.3f}")

In [ ]:
check_greater(bs8[512]["loss_fixed"], 10 * bs8[8]["loss_fixed"], "batch lớn + lr cố định: underfit trong cùng số epoch")
check_greater(bs8[512]["acc_scaled"], bs8[512]["acc_fixed"] + 0.05, "tăng lr theo batch size cứu được batch lớn")
check_equal(bs8[512]["steps"], 20 * math.ceil(len(Xt8) / 512), "số bước = epoch × số batch")

**Nhận xét mẫu:** Cùng số epoch, batch lớn → **ít bước cập nhật** hơn hẳn (60 so với 3 160). Với lr giữ nguyên, mỗi bước của batch lớn chỉ đi xa bằng một bước batch nhỏ (gradient trung bình có độ lớn tương tự, chỉ ít nhiễu hơn) → underfit. **Quy tắc tỉ lệ tuyến tính** (Goyal et al., 2017): nhân batch size lên $k$ thì nhân lr lên $k$ — giữ "quãng đường" mỗi epoch; với lr rất lớn cần **warmup**. Batch lớn tận dụng GPU tốt hơn (ít bước, mỗi bước song song nhiều mẫu); batch nhỏ có nhiễu gradient đóng vai trò regularization nhẹ. Quy tắc này chỉ đúng tới một giới hạn — batch quá lớn thì tăng lr thêm không còn hiệu quả.

## L6 — Thực tế

**Bài 9.** **Quy trình huấn luyện có kiểm chứng** (bài 04, mục 7) trên digits (train/val/test = 60/20/20, `random_state=0`, stratify):
1. **Overfit một batch**: 32 mẫu, 300 bước Adam lr 1e-3 → `one_batch_loss` < 0.01 (xác nhận pipeline đúng).
2. **LR range test** (SGD momentum 0.9, lr từ 1e-5 tới 10 theo hàm mũ, 150 batch) → `suggested_lr` = (lr có loss làm trơn nhỏ nhất) / 10.
3. Huấn luyện mô hình cuối (MLP 64 → 256 → 128 → 10, BatchNorm, dropout 0.2, SGD momentum với `suggested_lr`, cosine schedule, 40 epoch), **giữ checkpoint có val accuracy tốt nhất** → `test_acc9` ≥ 0.96.

In [ ]:
X9_tmp, X9_te, y9_tmp, y9_te = train_test_split(X_dg, y_dg, test_size=0.2, random_state=0, stratify=y_dg)
X9_tr, X9_va, y9_tr, y9_va = train_test_split(X9_tmp, y9_tmp, test_size=0.25, random_state=0, stratify=y9_tmp)
sc9 = StandardScaler().fit(X9_tr)
Xt9, Xv9, Xe9 = (torch.tensor(sc9.transform(a), dtype=torch.float32) for a in (X9_tr, X9_va, X9_te))
yt9, yv9, ye9 = (torch.tensor(a) for a in (y9_tr, y9_va, y9_te))


def make_model9(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(64, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 10))


### START CODE HERE ### (≈ 45 dòng)
# STARTER: # 1. overfit một batch: dùng một mạng phụ cùng kích thước nhưng KHÔNG có Dropout/BatchNorm (để loss có thể về ~0);
# STARTER: #    Adam lr 1e-3, 300 bước trên 32 mẫu train đầu tiên → one_batch_loss
# STARTER: # 2. LR range test trên make_model9() → suggested_lr
# STARTER: # 3. huấn luyện 40 epoch với CosineAnnealingLR, đánh giá val mỗi epoch (model.eval()), lưu state_dict tốt nhất → test_acc9
# STARTER: one_batch_loss, suggested_lr = None, None
# STARTER: best_val = None    # val accuracy tốt nhất trong 40 epoch
# STARTER: test_acc9 = None   # accuracy test của checkpoint tốt nhất
torch.manual_seed(0)
probe = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
opt_p = torch.optim.Adam(probe.parameters(), lr=1e-3)
xb, yb = Xt9[:32], yt9[:32]
for _ in range(300):
    opt_p.zero_grad()
    loss_p = nn.functional.cross_entropy(probe(xb), yb)
    loss_p.backward()
    opt_p.step()
one_batch_loss = loss_p.item()

model = make_model9()
opt = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)
g = torch.Generator().manual_seed(0)
lr_hist, loss_hist, smooth = [], [], None
for lr in np.logspace(-5, 1, 150):
    for group in opt.param_groups:
        group["lr"] = lr
    idx = torch.randint(0, len(Xt9), (32,), generator=g)
    model.train()
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(Xt9[idx]), yt9[idx])
    loss.backward()
    opt.step()
    smooth = loss.item() if smooth is None else 0.9 * smooth + 0.1 * loss.item()
    lr_hist.append(lr)
    loss_hist.append(smooth)
    if not math.isfinite(smooth) or smooth > 4 * min(loss_hist):
        break
suggested_lr = lr_hist[int(np.argmin(loss_hist))] / 10

model = make_model9()
opt = torch.optim.SGD(model.parameters(), lr=suggested_lr, momentum=0.9)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
g = torch.Generator().manual_seed(0)
best_val, best_state = -1.0, None
for epoch in range(40):
    model.train()
    perm = torch.randperm(len(Xt9), generator=g)
    for s in range(0, len(perm), 32):
        idx = perm[s:s + 32]
        opt.zero_grad()
        nn.functional.cross_entropy(model(Xt9[idx]), yt9[idx]).backward()
        opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        val_acc = (model(Xv9).argmax(1) == yv9).float().mean().item()
    if val_acc > best_val:
        best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    test_acc9 = (model(Xe9).argmax(1) == ye9).float().mean().item()
### END CODE HERE ###
print(f"overfit 1 batch: loss = {one_batch_loss:.5f} | lr gợi ý = {suggested_lr:.2e} | val tốt nhất = {best_val:.3f} | test = {test_acc9:.3f}")

In [ ]:
check_less(one_batch_loss, 0.01, "overfit được một batch")
check_true(1e-4 < suggested_lr < 1.0, "lr gợi ý hợp lý")
check_greater(test_acc9, 0.96, "accuracy test ≥ 0.96")

## L7 — Nâng cao

**Bài 10.** **Batch Norm backward from scratch.** Cho $\hat x = (x - \mu)/\sqrt{\sigma^2 + \epsilon}$, $y = \gamma\hat x + \beta$ (chuẩn hoá theo cột, $\sigma^2$ chia $n$). Viết `bn_forward(x, gamma, beta, eps)` → `(y, cache)` và `bn_backward(dy, cache)` → `(dx, dgamma, dbeta)` dùng công thức gọn:
$$d\hat x = dy\cdot\gamma, \qquad dx = \frac{1}{n\sqrt{\sigma^2 + \epsilon}}\Big(n\,d\hat x - \sum_i d\hat x_i - \hat x\sum_i d\hat x_i\hat x_i\Big)$$
Kiểm tra với `torch.autograd` và giải thích vì sao gradient $dx$ có tổng theo batch bằng 0.

In [ ]:
def bn_forward(x, gamma, beta, eps=1e-5):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return None, None   # y, cache = (x_hat, gamma, inv_std)
    mu = x.mean(axis=0)
    var = x.var(axis=0)
    inv_std = 1.0 / np.sqrt(var + eps)
    x_hat = (x - mu) * inv_std
    return gamma * x_hat + beta, (x_hat, gamma, inv_std)
    ### END CODE HERE ###


def bn_backward(dy, cache):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return None, None, None   # dx, dgamma, dbeta
    x_hat, gamma, inv_std = cache
    n = dy.shape[0]
    dgamma = (dy * x_hat).sum(axis=0)
    dbeta = dy.sum(axis=0)
    dx_hat = dy * gamma
    dx = inv_std / n * (n * dx_hat - dx_hat.sum(axis=0) - x_hat * (dx_hat * x_hat).sum(axis=0))
    return dx, dgamma, dbeta
    ### END CODE HERE ###

In [ ]:
x10 = rng.normal(3, 2, (16, 5))
gamma10, beta10 = rng.normal(size=5), rng.normal(size=5)
dy10 = rng.normal(size=(16, 5))
y10, cache10 = bn_forward(x10, gamma10, beta10)
dx10, dg10, db10 = bn_backward(dy10, cache10)
tx = torch.tensor(x10, requires_grad=True)
tg, tb = torch.tensor(gamma10, requires_grad=True), torch.tensor(beta10, requires_grad=True)
ty = nn.functional.batch_norm(tx, None, None, weight=tg, bias=tb, training=True, eps=1e-5)
ty.backward(torch.tensor(dy10))
check_close(y10, ty.detach().numpy(), tol=1e-10, message="forward khớp torch")
check_close(dx10, tx.grad.numpy(), tol=1e-10, message="dx khớp torch.autograd")
check_close(dg10, tg.grad.numpy(), tol=1e-10, message="dγ khớp")
check_close(db10, tb.grad.numpy(), tol=1e-10, message="dβ khớp")
check_close(dx10.sum(axis=0), np.zeros(5), tol=1e-10, message="tổng dx theo batch = 0")

**Nhận xét:** Đầu ra của BN không đổi nếu cộng cùng một hằng số vào mọi phần tử của cột (trung bình bị trừ đi) → đạo hàm theo hướng "dịch đều" bằng 0, tức $\sum_i dx_i = 0$; tương tự $\sum_i dx_i\hat x_i = 0$ (bất biến theo phép co giãn). Đây cũng là lý do bias của lớp Linear ngay trước BN là thừa. Gradient của mỗi mẫu phụ thuộc **cả batch** — nguồn gốc của hiện tượng "BN kém khi batch nhỏ".